# Session 39: Clustering with K-Means

Data Science with Python, Level 3, Stage 6: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Explain supervised and unsupervised learning.
2. Describe how K-Means finds clusters.
3. Fit K-Means and read its labels and centres.
4. Choose k with the elbow and silhouette methods.
5. Scale features before clustering.
6. Describe and name customer segments.

## Where We Are

Every model since Session 31 learned from a **target**: price, churn, fraud. That is **supervised** learning. Sometimes there is no target. A marketing team simply asks: what kinds of customers do we have?

Finding groups in data without a target is **unsupervised** learning, and the classic method is **K-Means clustering**. Today you will segment 250 practice mall customers by income and spending, choose the number of segments, and turn the clusters into a short description the marketing team can act on.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
mall = pd.read_csv(DATA_URL + "mall_customers.csv")

## 1. Looking for Groups

In [ ]:
print(mall.head())
print(mall.shape)

In [ ]:
sns.scatterplot(data=mall, x="annual_income_k", y="spending_score")
plt.title("Income and spending of mall customers")
plt.xlabel("Annual income (thousands)")
plt.show()

## 2. How K-Means Works

In [ ]:
X = mall[["annual_income_k", "spending_score"]]
kmeans = KMeans(n_clusters=5, n_init=10, random_state=0)
mall["segment"] = kmeans.fit_predict(X)

print(mall["segment"].value_counts().sort_index())
print(kmeans.cluster_centers_.round(1))

In [ ]:
sns.scatterplot(data=mall, x="annual_income_k", y="spending_score",
                hue="segment", palette="tab10")
centres = kmeans.cluster_centers_
plt.scatter(centres[:, 0], centres[:, 1], color="black", marker="X", s=200)
plt.title("Five customer segments")
plt.show()

## 3. Choosing k

In [ ]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    rows.append({"k": k, "inertia": km.inertia_,
                 "silhouette": silhouette_score(X, km.labels_)})
scores = pd.DataFrame(rows).set_index("k")
print(scores.round(3))

In [ ]:
scores["inertia"].plot(marker="o")
plt.title("Elbow method")
plt.ylabel("Inertia")
plt.show()

## 4. Scaling and Describing Segments

In [ ]:
print(mall[["age", "annual_income_k", "spending_score", "visits_per_month"]].std().round(1))

In [ ]:
cols = ["age", "annual_income_k", "spending_score"]
X_scaled = StandardScaler().fit_transform(mall[cols])

for k in [3, 4, 5, 6]:
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X_scaled)
    print(k, round(silhouette_score(X_scaled, km.labels_), 3))

In [ ]:
profile = mall.groupby("segment")[["age", "annual_income_k", "spending_score",
                                   "visits_per_month"]].mean().round(1)
profile["customers"] = mall["segment"].value_counts()
print(profile)

| Segment | Name | Profile | Action |
| --- | --- | --- | --- |
| 0 | Premium | Age 31, high income, high spending | Loyalty rewards, new collections |
| 1 | Mainstream | Age 45, middle income and spending | Seasonal offers |
| 2 | Young spenders | Age 25, low income, high spending | Student deals, social media |
| 3 | Budget | Age 50, low income, low spending | Discount days |
| 4 | Careful wealthy | Age 56, high income, low spending | Quality and service focus |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| No scaling | Clustering income in thousands next to visits per month | Standardise features first. |
| Labels as ranks | Treating segment 4 as better than segment 0 | Cluster numbers are just names. |
| Too many groups | Choosing k = 12 because inertia is lower | Use the elbow, silhouette and common sense. |
| Text columns | Passing `gender` straight into KMeans | Encode categories, or leave them for profiling. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Three Segments

Run K-Means with k = 3 on income and spending. Draw the scatter plot coloured by segment. Which of the five groups were merged?

In [ ]:
# Write your code below

### In-Class Exercise 2: Gender by Segment

Using the five segments, make a table with the share of female customers in each segment (`pd.crosstab` with `normalize="index"`). Is gender very different between segments?

In [ ]:
# Write your code below

### In-Class Exercise 3: Assign New Customers

Three new customers have incomes 30, 60 and 95 thousand and spending scores 85, 50 and 15. Use the fitted five-cluster model's `predict` to assign each one to a segment.

In [ ]:
# Write your code below

### In-Class Exercise 4: Silhouette Chart

Plot the silhouette score against k from 2 to 10 for the scaled features age, income and spending. Mark the best k with a vertical line.

In [ ]:
# Write your code below

## Summary

- Unsupervised learning finds structure without a target.
- K-Means assigns points to the nearest centre and moves the centres.
- `fit_predict` gives labels; `cluster_centers_` gives the centres.
- Choose k with the elbow, the silhouette and business sense.
- Scale features so that no single unit dominates the distance.
- Profile and name segments so others can act on them.

## What's Next

Back to supervised learning, with a hard case: only 1.8 percent of card payments are fraud. Next session you will see why imbalance breaks naive models and fix it with class weights, resampling and better thresholds.